In [ ]:
import json
import time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

In [ ]:
API_KEY = "--"

MODEL_NAME = "llama-3.3-70b-versatile"

BASE_URL = "https://api.groq.com/openai/v1"

RAW_OUTPUT_PATH = "llm_synthetic_raw.jsonl"
SYNTHETIC_OUTPUT_PATH = "llm_synthetic_final.jsonl"

N_BATCHES = 49
EXAMPLES_PER_BATCH = 15

MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 2

In [3]:
from category_config import PRODUCT_CATEGORIES, CATEGORY_ASPECT_HINTS

In [ ]:
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
)

In [5]:
SYSTEM_PROMPT = """
شما یک تولیدکننده داده آموزشی باکیفیت برای یک سیستم
Aspect-Based Sentiment Analysis (ABSA) به زبان فارسی هستید.

وظیفه شما تولید نظرهای طبیعی و متنوع کاربران ایرانی درباره محصولات فروشگاهی است.

قوانین:

1. متن‌ها باید فارسی طبیعی باشند و شبیه نظر واقعی کاربران در فروشگاه‌های آنلاین ایرانی باشند.
2. از تکرار متن‌ها خودداری کنید.
3. هر review_text باید حداقل یک aspect داشته باشد.
4. term باید دقیقاً substring متن review_text باشد.
5. term نباید بازنویسی، خلاصه یا اصلاح شود.
6. polarity فقط باید یکی از مقادیر زیر باشد:
   positive
   negative
   neutral
7. یک review_text می‌تواند چند aspect داشته باشد.
8. برای هر aspect باید polarity مناسب همان aspect تعیین شود.
9. فقط JSON معتبر تولید کنید.
""".strip()

In [6]:
def build_user_prompt(category, n_pos, n_neg, n_neu):

    category_aspects = CATEGORY_ASPECT_HINTS.get(category, [])

    return f"""
برای دسته محصول زیر داده آموزشی ABSA تولید کن:

دسته محصول:
{category}

دقیقاً تولید کن:

- {n_pos} نظر مثبت
- {n_neg} نظر منفی
- {n_neu} نظر خنثی

مجموعاً:
{n_pos + n_neg + n_neu} نظر

هر review_text:

- باید بین 20 تا 50 کلمه باشد.
- باید فارسی طبیعی داشته باشد.
- باید شبیه نظر واقعی کاربران ایرانی در سایت‌های فروشگاهی باشد.
- می‌تواند رسمی یا محاوره‌ای باشد.
- باید حداقل یک aspect داشته باشد.
- می‌تواند بیش از یک aspect داشته باشد.
- از 50 درصد aspect ها منفی باشند و 40 درصد آنها خنثی و تنها 10 درصد مثبت باشند

از این جنبه‌ها به صورت متنوع استفاده کن:

{", ".join(category_aspects)}

قانون بسیار مهم:

مقدار term باید دقیقاً همان عبارت موجود در review_text باشد.

مثال:

اگر متن باشد:

"باتری گوشی خیلی زود خالی می‌شود ولی کیفیت صفحه نمایش عالی است"

خروجی صحیح:

{{
    "review_text": "باتری گوشی خیلی زود خالی می‌شود ولی کیفیت صفحه نمایش عالی است",
    "aspects": [
        {{
            "term": "باتری",
            "polarity": "negative"
        }},
        {{
            "term": "صفحه نمایش",
            "polarity": "positive"
        }}
    ]
}}

فقط JSON معتبر تولید کن.

ساختار خروجی دقیقاً باید این باشد:

{{
    "examples": [
        {{
            "review_text": "...",
            "aspects": [
                {{
                    "term": "...",
                    "polarity": "positive"
                }}
            ]
        }}
    ]
}}
""".strip()


In [7]:
def extract_examples(parsed):

    if isinstance(parsed, dict):
        examples = parsed.get("examples", [])
        return examples if isinstance(examples, list) else []

    if isinstance(parsed, list):
        return parsed

    return []

In [ ]:
def generate_batch(batch_idx, category):

    n_each = EXAMPLES_PER_BATCH // 3

    n_pos = n_each
    n_neg = n_each
    n_neu = EXAMPLES_PER_BATCH - n_pos - n_neg

    user_prompt = build_user_prompt(
        category=category,
        n_pos=n_pos,
        n_neg=n_neg,
        n_neu=n_neu,
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    for attempt in range(1, MAX_RETRIES + 1):

        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=0.65,
                max_tokens=4000,
                response_format={"type": "json_object"},
            )

            response_text = response.choices[0].message.content

            if not response_text:
                raise ValueError("مدل خروجی خالی برگرداند.")

            response_text = response_text.strip()

            parsed = json.loads(response_text)

            examples = extract_examples(parsed)

            if not examples:
                print(
                    f"⚠️ batch {batch_idx} تلاش {attempt}: "
                    f"هیچ examples معتبری دریافت نشد."
                )

                time.sleep(RETRY_DELAY_SECONDS)

                continue

            return {
                "batch": batch_idx,
                "category": category,
                "raw": response_text,
                "examples": examples,
            }

        except json.JSONDecodeError as e:

            print(
                f"⚠️ batch {batch_idx} تلاش {attempt}: "
                f"JSON نامعتبر بود -> {e}"
            )

        except Exception as e:

            print(
                f"⚠️ خطا در batch {batch_idx} "
                f"تلاش {attempt}: {e}"
            )

        if attempt < MAX_RETRIES:

            sleep_time = RETRY_DELAY_SECONDS * attempt

            time.sleep(sleep_time)

    return {
        "batch": batch_idx,
        "category": category,
        "raw": "",
        "examples": [],
    }

In [10]:
def validate_examples(examples, seen_texts):

    rows = []

    kept = 0
    dropped = 0

    valid_polarities = {
        "positive",
        "negative",
        "neutral",
    }

    for example in examples:

        if not isinstance(example, dict):
            dropped += 1
            continue

        text = str(
            example.get("review_text", "")
        ).strip()

        aspects = example.get(
            "aspects",
            []
        )

        if not text:

            dropped += 1

            continue

        if text in seen_texts:

            dropped += 1

            continue

        if not isinstance(aspects, list) or not aspects:

            dropped += 1

            continue

        valid_rows_for_text = []

        for aspect in aspects:

            if not isinstance(aspect, dict):
                dropped += 1
                continue

            term = str(
                aspect.get("term", "")
            ).strip()

            polarity = str(
                aspect.get("polarity", "")
            ).strip().lower()

            if not term:

                dropped += 1

                continue

            if polarity not in valid_polarities:

                dropped += 1

                continue

            start_idx = text.find(term)

            if start_idx == -1:

                dropped += 1

                continue

            end_idx = start_idx + len(term)

            valid_rows_for_text.append(
                {
                    "review_text": text,
                    "term": term,
                    "polarity": polarity,
                    "asp_start": start_idx,
                    "asp_end": end_idx,
                }
            )

        if valid_rows_for_text:

            seen_texts.add(text)

            rows.extend(valid_rows_for_text)

            kept += len(valid_rows_for_text)

    return rows, kept, dropped

In [ ]:
print("Testing Groq connection...")

test_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": 'Return exactly this JSON: {"status": "OK"}',
        }
    ],

    response_format={
        "type": "json_object"
    },
    temperature=0,
)

print(
    "Connection successful:",
    test_response.choices[0].message.content
)


Testing Groq connection...


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/migrate-to-interactions).', 'status': 'NOT_FOUND'}}

In [12]:
synthetic_rows = []

seen_texts = set()

total_kept = 0
total_dropped = 0
successful_batches = 0


print(
    f"\nStarting synthetic ABSA data generation..."
)

print(
    f"Model: {MODEL_NAME}"
)

print(
    f"Batches: {N_BATCHES}"
)

print(
    f"Examples per batch: {EXAMPLES_PER_BATCH}\n"
)


with open(
    RAW_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as raw_file:

    for batch_idx in tqdm(
        range(N_BATCHES),
        desc="Generating batches",
    ):

        category = PRODUCT_CATEGORIES[
            batch_idx % len(PRODUCT_CATEGORIES)
        ]

        result = generate_batch(
            batch_idx=batch_idx,
            category=category,
        )

        if result["raw"]:

            raw_file.write(
                json.dumps(
                    {
                        "batch": result["batch"],
                        "category": result["category"],
                        "raw": result["raw"],
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )

            raw_file.flush()

        if not result["examples"]:

            print(
                f"❌ batch {batch_idx}: "
                f"no valid LLM output"
            )

            continue

        successful_batches += 1

        rows, kept, dropped = validate_examples(
            examples=result["examples"],
            seen_texts=seen_texts,
        )

        synthetic_rows.extend(rows)

        total_kept += kept
        total_dropped += dropped

        print(
            f"Batch {batch_idx + 1}/{N_BATCHES} | "
            f"Category: {category} | "
            f"Rows kept: {kept} | "
            f"Rows dropped: {dropped}"
        )

        time.sleep(1)


Starting synthetic ABSA data generation...
Model: gemini-2.5-flash
Batches: 49
Examples per batch: 15



Generating batches:   0%|          | 0/49 [00:00<?, ?it/s]

⚠️ خطا در batch 0 تلاش 1: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/migrate-to-interactions).', 'status': 'NOT_FOUND'}}
⚠️ خطا در batch 0 تلاش 2: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/migrate-to-interactions).', 'status': 'NOT_FOUND'}}


Generating batches:   0%|          | 0/49 [00:08<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
columns = [
    "review_text",
    "term",
    "polarity",
    "asp_start",
    "asp_end",
]

df_final = pd.DataFrame(
    synthetic_rows,
    columns=columns,
)

In [ ]:
if not df_final.empty:

    df_final.to_json(
        SYNTHETIC_OUTPUT_PATH,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    print("\n" + "=" * 60)
    print("GENERATION FINISHED")
    print("=" * 60)

    print(
        f"Successful batches: "
        f"{successful_batches}/{N_BATCHES}"
    )

    print(
        f"Valid ABSA rows: "
        f"{len(df_final)}"
    )

    print(
        f"Kept aspect rows: "
        f"{total_kept}"
    )

    print(
        f"Dropped invalid items: "
        f"{total_dropped}"
    )

    print(
        "\nClass distribution:"
    )

    print(
        df_final["polarity"].value_counts()
    )

    print(
        f"\nFinal dataset saved to:\n"
        f"{SYNTHETIC_OUTPUT_PATH}"
    )

    print(
        f"\nRaw model outputs saved to:\n"
        f"{RAW_OUTPUT_PATH}"
    )

else:

    print("\n❌ No valid synthetic data was generated.")
    print(
        "Check the API key, model availability, "
        "or batch errors."
    )


GENERATION FINISHED
Successful batches: 35/47
Valid ABSA rows: 845
Kept aspect rows: 845
Dropped invalid items: 32

Class distribution:
polarity
positive    373
negative    308
neutral     164
Name: count, dtype: int64

Final dataset saved to:
llm_synthetic_final.jsonl

Raw model outputs saved to:
llm_synthetic_raw.jsonl
